# Stg Layer — comments (Incremental)

Read **comments** từ Bronze (`prod.bronze.comments_raw`, append-only copy of Lakebase Postgres) → MERGE 1:1 vào Delta staging table. Comments được app fetch on-demand (per video, qua nút bấm) nên nguồn mới, chưa có nhiều rows — logic giống các table khác.

**Incremental Flow:**
1. Lấy watermark (`MAX(COALESCE(updated_at, created_at))`) từ Delta table
2. SELECT từ `prod.bronze.comments_raw` (Bronze): chỉ rows có `COALESCE(updated_at, created_at) > watermark`
3. MERGE vào `prod.silver.stg_comments` (UPSERT on `comment_id`)
4. Verify row count

**Kiến trúc:** Databricks App → user search/save YouTube data → INSERT/UPDATE vào Postgres → `Analytic/bronze/ingest_raw` (append vào Bronze) → Job notebook (this) → SELECT Bronze (incremental) → MERGE vào Delta (stg layer, 1:1 với nguồn).


> **Source change (sub-task 3):** reads `prod.bronze.comments_raw` instead of `lakebase_pg` and writes `prod.silver.stg_comments`. Bronze keeps every read, so rows are deduped to the latest read per grain key before the MERGE.

In [0]:
# --- Config ---
SOURCE_TABLE = "prod.bronze.comments_raw"  # Bronze: append-only copy of every Postgres read
DELTA_TABLE = "prod.silver.stg_comments"
MERGE_KEY = "comment_id"

print(f"Bronze source: {SOURCE_TABLE}")
print(f"Delta target: {DELTA_TABLE}")

In [0]:
# --- Get watermark from Delta table (last successful ETL) ---
try:
    watermark_row = spark.sql(f"SELECT MAX(COALESCE(updated_at, created_at)) AS wm FROM {DELTA_TABLE}").collect()[0]
    watermark = watermark_row["wm"] if watermark_row else None
except Exception:
    watermark = None

if watermark:
    # Incremental: only rows changed after watermark
    # Use Spark SQL — timestamps stay in Spark's timezone (UTC), no JDBC mismatch
    src_df = spark.sql(f"""
        SELECT * FROM {SOURCE_TABLE}
        WHERE COALESCE(updated_at, created_at) > '{watermark.strftime("%Y-%m-%d %H:%M:%S.%f")}'
    """)
    print(f"💧 Watermark: {watermark}")
    print("Incremental load: rows after watermark")
else:
    # Full load (first run or Delta empty)
    src_df = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")
    print("📦 Full load (no watermark — first run)")

src_count = src_df.count()
print(f"Read {src_count} rows from Bronze ✅")
if src_count > 0:
    display(src_df.limit(10))

In [0]:
# --- Add etl_date and op_type to source data ---
from pyspark.sql.functions import current_timestamp, lit, col
# --- Keep the latest Bronze read per grain key (Bronze holds every read) ---
from pyspark.sql import Window
from pyspark.sql.functions import row_number
latest_read = Window.partitionBy("comment_id").orderBy(col("ingest_ts").desc())
src_df = (
    src_df.withColumn("_rn", row_number().over(latest_read))
    .filter("_rn = 1")
    .drop("_rn", "ingest_ts", "ingest_date")
)
src_df = src_df.withColumn("etl_date", current_timestamp())
src_df = src_df.withColumn("op_type", lit("INSERT"))

# --- Create Delta table if not exists (1:1 with source + ETL metadata) ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
        comment_id          STRING,
        video_id            STRING,
        author_display_name STRING,
        author_channel_id   STRING,
        text                STRING,
        like_count          BIGINT,
        published_at        TIMESTAMP,
        total_reply_count   INT,
        created_at          TIMESTAMP,
        updated_at          TIMESTAMP,
        etl_date       TIMESTAMP,
        op_type        STRING
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"Delta table {DELTA_TABLE} ready ✅")

# --- MERGE via SQL (compact, op_type tracking) ---
src_df.createOrReplaceTempView("src_source")

spark.sql(f"""
    MERGE INTO {DELTA_TABLE} AS t
    USING src_source AS s ON t.comment_id = s.comment_id
    WHEN MATCHED THEN UPDATE SET
        video_id=s.video_id,
        author_display_name=s.author_display_name,
        author_channel_id=s.author_channel_id,
        text=s.text,
        like_count=s.like_count,
        published_at=s.published_at,
        total_reply_count=s.total_reply_count,
        updated_at=s.updated_at,
        etl_date=s.etl_date, op_type='UPDATE'
    WHEN NOT MATCHED THEN INSERT *
""")

merged_count = spark.sql(f"SELECT COUNT(*) FROM {DELTA_TABLE}").collect()[0][0]
print(f"✅ Merged into {DELTA_TABLE}: {merged_count} total rows")

In [0]:
# --- Verify ---
total_delta = spark.sql(f"SELECT COUNT(*) as cnt FROM {DELTA_TABLE}").collect()[0]["cnt"]
last_etl = spark.sql(f"SELECT MAX(etl_date) as etl FROM {DELTA_TABLE}").collect()[0]["etl"]

op_counts = spark.sql(f"SELECT op_type, COUNT(*) as cnt FROM {DELTA_TABLE} GROUP BY op_type").collect()
op_breakdown = ", ".join(f"{r['op_type']}: {r['cnt']}" for r in op_counts)

print(f"📊 Total rows in {DELTA_TABLE}: {total_delta}")
print(f"📊 Last ETL date: {last_etl}")
print(f"📊 Op type breakdown: {op_breakdown}")
print(f"📊 Rows loaded this run: {src_count}")
print("\n--- stg_comments ETL complete ---")